# Problem 624 - Two Heads Are Better Than One
An unbiased coin is tossed repeatedly until two consecutive heads are obtained. Suppose these occur on the $(M-1)$ th and $M$ th toss.

Let $P(n)$ be the probability that $M$ is divisible by $n$. For example, the outcomes HH, HTHH, and THTTHH all count towards $P(2)$, but THH and HTTHH do not.

You are given that $P(2) =\frac 3 5$ and $P(3)=\frac 9 {31}$. Indeed, it can be shown that $P(n)$ is always a rational number.

For a prime $p$ and a fully reduced fraction $\frac a b$, define $Q(\frac a b,p)$ to be the smallest positive $q$ for which $a \equiv b q \pmod{p}$.

For example $Q(P(2), 109) = Q(\frac 3 5, 109) = 66$, because $5 \cdot 66 = 330 \equiv 3 \pmod{109}$ and $66$ is the smallest positive such number.

Similarly $Q(P(3),109) = 46$.

Find $Q(P(10^{18}),1\,000\,000\,009)$.


## Solution.
It can be easily shown that $P(n) = \sum_{k=1}^\infty \frac{1}{2^{kn}}  F_{kn-1} $, which can be caculated as

$$ P(n) =\frac{2^{n}F_{n-1}-(-1)^{n}}{4^{n}-2^{n}F_{n-1}-2^{n}F_{n+1}+(-1)^{n}},$$

where $F$ is Fibonnaci sequence ($F_0=0, F_1=1$). Let
$$A_n = 2^{n}F_{n-1}-(-1)^{n}$$
and
$$B_n = 4^{n}-2^{n}F_{n-1}-2^{n}F_{n+1}+(-1)^{n}.$$
Then $q = (A_n \mod p) * (B_n \mod p)^{p-2} \mod p,$
where we use LFT for the inverse. Note that we do not need to reduce $A_n$ and $B_n$ as it will cancel out in multipliaction by inverse.

In [1]:
from functools import cache

In [2]:
@cache
def F(n):
    if n == 0: return 0
    if n == 1: return 1
    return F(n-1) + F(n-2)

In [3]:
def P(n):
    return (2**n*F(n-1)-(-1)**n) / (4**n - 2**n*F(n-1)-2**n *F(n+1)+(-1)**n)

In [4]:
P(2) == 0.6, P(3) == 9/31

(True, True)

In [6]:
def mod_pow(a, n, p):
    '''Finds a^n mod p'''
    result = 1
    
    a = a % p
    while n > 0:
        if n % 2 == 1:
            result = (result * a) % p
        a = (a * a) % p
        n = n // 2
    return result

In [7]:
def multiply_matrices(A, B, p):
    C = [[0, 0], [0, 0]]
    
    for i in range(2):
        for j in range(2):
            C[i][j] = sum(A[i][k] * B[k][j] for k in range(2)) % p
    return C

def fibonacci_modulo(n, p):
    if n == 0:
        return 0
    
    M = [[1, 1], [1, 0]]
    result = [[1, 0], [0, 1]]
    
    power = n
    while power > 0:
        if power % 2 == 1:
            result = multiply_matrices(result, M, p)
        M = multiply_matrices(M, M, p)
        power //= 2
        
    return result[0][1]

In [9]:
def Q(n, p):
    A = (mod_pow(2, n, p) * fibonacci_modulo(n-1, p) - (-1 if n%2 == 1 else 1)) % p
    B = (-A - mod_pow(2, n, p) * fibonacci_modulo(n+1, p) + mod_pow(4, n, p)) % p

    return (A * mod_pow(B, p-2, p)) % p

In [10]:
Q(2, 109), Q(3, 109)

(66, 46)

In [11]:
Q(10**18, 1_000_000_009)

984524441